# Provider NPI Registry Scraper

Queries the public [CMS NPI Registry](https://npiregistry.cms.hhs.gov/) to retrieve
provider specialty and demographic information for attending physicians in the study cohort.

**Input:** `no_push/provider_npis.csv` — de-identified provider NPI list (gitignored, site-specific)  
**Output:** `no_push/npi_scrapped.csv` — scraped registry data (gitignored, not distributed)

The NPI registry is a public dataset (https://npiregistry.cms.hhs.gov/). Output is stored
locally and excluded from the repository via `.gitignore`.

In [ ]:
#pip install webdriver-manager

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import numpy as np

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

import re

In [ ]:
# Set up the Chrome driver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run in headless mode (no GUI)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [ ]:
npis = pd.read_csv('no_push/provider_npis.csv')
npis

In [ ]:
taxonomy_codes = pd.read_csv('nucc_taxonomy_250.csv')[['Code', 'Classification', 'Specialization']]
taxonomy_codes["Primary Specialization"] = taxonomy_codes["Classification"].where(taxonomy_codes["Specialization"].isna(), taxonomy_codes["Classification"] + " - " + taxonomy_codes["Specialization"])
taxonomy_codes.columns = ['Primary Taxonomy', 'Classification', 'Specialization', 'Primary Specialization']
taxonomy_map = taxonomy_codes.set_index('Primary Taxonomy')['Primary Specialization'].to_dict()
taxonomy_map

In [ ]:
npi_series = npis['prov_npi_shifted'].astype(str)
npi_series = npi_series[~npi_series.isin(['nan', 'NaN', 'None', ''])]
npi_list = [s[:-2] if s.endswith('.0') else s for s in npi_series]

npi_element_list, name_element_list, gender_element_list, enumeration_element_list, primary_taxonomy_element_list, secondary_taxonomy_elements_list = [],[],[],[],[],[]

wait = WebDriverWait(driver, 10)

def safe_find_text(xpath):
    try:
        return driver.find_element(By.XPATH, xpath).text
    except NoSuchElementException:
        return None

for i, npi in enumerate(npi_list):
    print(f'{i+1} of {len(npi_list)}')
    url = f"https://npiregistry.cms.hhs.gov/provider-view/{npi}"
    driver.get(url)

    try:
        wait.until(EC.presence_of_element_located((By.XPATH, "//blockquote/p[@tabindex='0']")))
    except TimeoutException:
        npi_element_list.append(npi)
        name_element_list.append(None)
        gender_element_list.append(None)
        enumeration_element_list.append(None)
        primary_taxonomy_element_list.append(None)
        secondary_taxonomy_elements_list.append([])
        continue

    name_element = safe_find_text("//blockquote/p[@tabindex='0']")
    if name_element:
        name_element = name_element.replace(' MD', '').replace(' M.D.', '').replace(' Dr.', '')

    gender_element = safe_find_text("//*[contains(text(), 'Sex:')]")
    if gender_element:
        gender_element = gender_element.replace('Sex: ','')

    enumeration_element = safe_find_text("//td[contains(text(), 'Enumeration Date')]/following-sibling::td")

    taxonomy_text = safe_find_text("//td[contains(text(), 'Taxonomy')]/following-sibling::td")
    primary_taxonomy_element = None
    secondary_taxonomy_elements = []
    if taxonomy_text:
        primary_matches = re.findall(r'\nYes\s+(\w+)', taxonomy_text)
        if primary_matches:
            primary_taxonomy_element = primary_matches[0]

        secondary_taxonomy_elements = re.findall(r'\nNo\s+(\w+)', taxonomy_text)

    if primary_taxonomy_element in secondary_taxonomy_elements:
        secondary_taxonomy_elements.remove(primary_taxonomy_element)

    secondary_taxonomy_elements = np.unique(secondary_taxonomy_elements).tolist()

    npi_element_list.append(npi)
    name_element_list.append(name_element)
    gender_element_list.append(gender_element)
    enumeration_element_list.append(enumeration_element)
    primary_taxonomy_element_list.append(primary_taxonomy_element)
    secondary_taxonomy_elements_list.append(secondary_taxonomy_elements)

df = {
    'NPI': npi_element_list,
    'Name': name_element_list,
    'Sex': gender_element_list,
    'Enumeration Date': enumeration_element_list,
    'Primary Taxonomy': primary_taxonomy_element_list,
    'Secondary Taxonomy': secondary_taxonomy_elements_list
}

df = pd.DataFrame(df)
df = df.merge(taxonomy_codes[['Primary Taxonomy', 'Primary Specialization']], on='Primary Taxonomy', how='left')

expanded_df = df['Secondary Taxonomy'].apply(lambda x: pd.Series(x[:5]))
expanded_df.columns = [f'Secondary Taxonomy {i+1}' for i in range(expanded_df.shape[1])]
df_final = pd.concat([df, expanded_df], axis=1).drop(columns=['Secondary Taxonomy'])

df_final['Secondary Specialization 1'] = df_final['Secondary Taxonomy 1'].map(taxonomy_map)
df_final['Secondary Specialization 2'] = df_final['Secondary Taxonomy 2'].map(taxonomy_map)
df_final['Secondary Specialization 3'] = df_final['Secondary Taxonomy 3'].map(taxonomy_map)
df_final['Secondary Specialization 4'] = df_final['Secondary Taxonomy 4'].map(taxonomy_map)

df_final = df_final[['NPI', 'Name', 'Sex', 'Enumeration Date',
                     'Primary Specialization', 'Secondary Specialization 1', 'Secondary Specialization 2', 'Secondary Specialization 3', 'Secondary Specialization 4',
                     'Primary Taxonomy', 'Secondary Taxonomy 1', 'Secondary Taxonomy 2', 'Secondary Taxonomy 3', 'Secondary Taxonomy 4'
                     ]]
df_final.to_csv('no_push/npi_scrapped.csv', index=False)

df_final

In [ ]:
npis['prov_npi_clean'] = npis['prov_npi_shifted'].astype(str)
npis['prov_npi_clean'] = npis['prov_npi_clean'].replace(['nan', 'NaN', 'None', ''], np.nan)
npis['prov_npi_clean'] = npis['prov_npi_clean'].str.replace(r'\.0$', '', regex=True)
data = npis.merge(df_final, left_on='prov_npi_clean', right_on='NPI', how='left')
data


In [ ]:
# Table 1 (single table)
as_of = pd.to_datetime('2025-12-01')
data['Enumeration Date'] = pd.to_datetime(data['Enumeration Date'], errors='coerce')
data['Experience (Yr)'] = (as_of - data['Enumeration Date']).dt.days / 365.25

n_total = data['NPI'].nunique()

sex_counts = data['Sex'].value_counts(dropna=True)
sex_pct = (sex_counts / sex_counts.sum() * 100).round(1)

exp = data['Experience (Yr)'].dropna()
exp_summary = f"{exp.median():.1f} [{exp.quantile(0.25):.1f}, {exp.quantile(0.75):.1f}]" if len(exp) else None

spec_counts = data['Primary Specialization'].value_counts(dropna=True)
spec_pct = (spec_counts / spec_counts.sum() * 100).round(1)

rows = []
rows.append({'Provider Characteristics': 'n', 'Value': n_total})

rows.append({'Provider Characteristics': 'Sex', 'Value': None})
for k, v in sex_counts.items():
    pct = sex_pct.get(k, None)
    rows.append({
        'Provider Characteristics': str(k),
        'Value': f"{int(v)} ({pct:.1f})" if pct is not None else str(int(v))
    })

rows.append({'Provider Characteristics': 'Experience (Yr), median [Q1,Q3]', 'Value': exp_summary})

rows.append({'Provider Characteristics': 'Specialty, n (%)', 'Value': None})
for k, v in spec_counts.items():
    pct = spec_pct.get(k, None)
    rows.append({
        'Provider Characteristics': str(k),
        'Value': f"{int(v)} ({pct:.1f})" if pct is not None else str(int(v))
    })

table1 = pd.DataFrame(rows)
table1.to_csv('no_push/table1_provider_characteristics.csv', index=False)
table1